<a href="https://colab.research.google.com/github/musowjanya/Natural-Language-Processing/blob/main/BasicTransformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

In [2]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(Encoder, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.multihead_attn(x, x, x)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x

In [3]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(Decoder, self).__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.encoder_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, enc_output):
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm(x + attn_output)
        attn_output, _ = self.encoder_attn(x, enc_output, enc_output)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x

In [4]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(Transformer, self).__init__()
        self.encoder = Encoder(embed_dim, num_heads)
        self.decoder = Decoder(embed_dim, num_heads)
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, src, tgt):
        enc_output = self.encoder(src)
        dec_output = self.decoder(tgt, enc_output)
        output = self.fc(dec_output)
        return output

In [5]:
# Example usage
embed_dim = 64
num_heads = 8
transformer_model = Transformer(embed_dim, num_heads)
src = torch.rand(10, 32, embed_dim)  # (sequence_length, batch_size, embed_dim)
tgt = torch.rand(10, 32, embed_dim)  # (sequence_length, batch_size, embed_dim)
output = transformer_model(src, tgt)
print("Output shape:", output.shape)

Output shape: torch.Size([10, 32, 64])


#Transformer with BERT based Embeddings

In [6]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
source_sentences = ["Hello world", "Machine learning is great"]
target_sentences = ["Bonjour le monde", "L'apprentissage automatique est génial"]

In [8]:
source_inputs = tokenizer(source_sentences, return_tensors='pt', padding=True, truncation=True, max_length=512)
target_inputs = tokenizer(target_sentences, return_tensors='pt', padding=True, truncation=True, max_length=512)

In [9]:
# Get BERT embeddings
with torch.no_grad():
  source_outputs = bert_model(**source_inputs)
  target_outputs = bert_model(**target_inputs)
  source_last_hidden_states = source_outputs.last_hidden_state
  target_last_hidden_states = target_outputs.last_hidden_state

In [10]:
class Encoder(nn.Module):
    def __init__(self, bert_model, embed_dim, num_heads):
        super().__init__()
        self.bert = bert_model
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            x = outputs.last_hidden_state
        attn_output, _ = self.multihead_attn(x, x, x)
        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x

In [11]:
class Decoder(nn.Module):
    def __init__(self, bert_model, embed_dim, num_heads):
        super(Decoder, self).__init__()
        self.bert = bert_model
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.encoder_attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, input_ids, attention_mask, enc_output):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            x = outputs.last_hidden_state
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm(x + attn_output)

        # enc_output = enc_output.transpose(0, 1)
        # enc_output = enc_output[:, 0]

        attn_output, _ = self.encoder_attn(x, enc_output, enc_output)

        x = self.norm(x + attn_output)
        x = self.norm(x + self.linear(x))
        return x

In [12]:
# generated = [BOS]

# for _ in range(max_len):
#     decoder_input = torch.tensor(generated).unsqueeze(0)

#     logits = model(src, decoder_input)   # forward pass
#     next_token = logits.argmax(dim=-1)

#     generated.append(next_token.item())

#     if next_token == EOS:
#         break